# 04 – Hypothesis testing
Formal statistical tests for the research questions.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats.contingency import association
from pathlib import Path

clean_folder = Path("../data/clean")
daily = pd.read_csv(clean_folder / "daily_summer.csv", parse_dates=["date"])

daily = daily.sort_values("date").reset_index(drop=True)
daily["prev_day_hot"] = daily.groupby("year")["hot_day"].shift(1)

summer = daily[(daily["in_summer"] == 1) & daily["ems_missions"].notna()].copy()
summer["ems_index"] = summer["ems_missions"] / summer.groupby("year")["ems_missions"].transform("mean") * 100
summer["cpr_index"] = summer["cpr_alert_missions"] / summer.groupby("year")["cpr_alert_missions"].transform("mean") * 100
summer["temp_band"] = pd.cut(summer["tmax"], bins=[0, 20, 22, 24, 26, 28, 30, 32, 34, 45], right=False,
                             labels=["<20", "20–22", "22–24", "24–26", "26–28", "28–30", "30–32", "32–34", "34+"])

alpha = 0.05
print(summer.shape)

(822, 17)


In [2]:
def compare_groups(group_a, group_b, name_a, name_b, alternative="greater"):
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False, alternative=alternative)

    diff = group_a.mean() - group_b.mean()
    se = np.sqrt(group_a.var(ddof=1) / len(group_a) + group_b.var(ddof=1) / len(group_b))
    ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

    print(f"{name_a}: n = {len(group_a)}, mean = {group_a.mean():.1f}")
    print(f"{name_b}: n = {len(group_b)}, mean = {group_b.mean():.1f}")
    print(f"Difference: {diff:.1f} index points, 95% CI [{ci_low:.1f}, {ci_high:.1f}]")
    print(f"t = {t_stat:.2f}, p-value = {p_value:.2e}")

    if p_value < alpha:
        print("Decision: reject H0")
    else:
        print("Decision: we cannot reject H0")

### Test 1 – Hot days vs. normal days
H0: mean EMS demand on hot days ≤ normal days
H1: mean EMS demand on hot days > normal days (one-sided, α = 0.05)

In [3]:
hot = summer[summer["hot_day"] == 1]["ems_index"]
normal = summer[summer["hot_day"] == 0]["ems_index"]

print("Levene p-value:", round(stats.levene(hot, normal).pvalue, 3))
print("Shapiro p-value, hot days:   ", round(stats.shapiro(hot).pvalue, 3))
print("Shapiro p-value, normal days:", round(stats.shapiro(normal).pvalue, 3))

Levene p-value: 0.731
Shapiro p-value, hot days:    0.038
Shapiro p-value, normal days: 0.077


In [4]:
compare_groups(hot, normal, "Hot days", "Normal days")

pooled_sd = np.sqrt(((len(hot) - 1) * hot.var(ddof=1) + (len(normal) - 1) * normal.var(ddof=1)) / (len(hot) + len(normal) - 2))
print(f"Cohen's d = {(hot.mean() - normal.mean()) / pooled_sd:.2f}")

print("Mann-Whitney U p-value:", f"{stats.mannwhitneyu(hot, normal, alternative='greater').pvalue:.2e}")

Hot days: n = 155, mean = 105.5
Normal days: n = 667, mean = 98.7
Difference: 6.7 index points, 95% CI [5.7, 7.8]
t = 12.20, p-value = 5.26e-27
Decision: reject H0
Cohen's d = 1.09
Mann-Whitney U p-value: 1.46e-28


### Test 2 – Consistency across summers (sign test)
Consecutive hot days are not independent (heatwaves), so the t-test p-value is likely too optimistic.
A simpler check treats each summer as one observation.
H0: hot days are busier in only half of the summers (pure chance)
H1: hot days are busier in more summers than chance would explain

In [5]:
per_year = summer.groupby(["year", "hot_day"])["ems_index"].mean().unstack()
per_year.columns = ["normal_days", "hot_days"]
per_year["hot_busier"] = per_year["hot_days"] > per_year["normal_days"]
print(per_year.round(1))

k = per_year["hot_busier"].sum()
n = len(per_year)
p_sign = stats.binomtest(k, n, p=0.5, alternative="greater").pvalue
print(f"\nHot days busier in {k} of {n} summers, p = {p_sign:.4f}")

      normal_days  hot_days  hot_busier
year                                   
2018         97.9     105.7        True
2019         97.8     105.9        True
2020         98.6     106.6        True
2021         99.1     105.3        True
2022         99.0     103.3        True
2023         99.1     106.8        True
2024         99.1     105.0        True
2025         99.4     104.2        True
2026         98.2     106.4        True

Hot days busier in 9 of 9 summers, p = 0.0020


### Test 3 – Correlation between temperature and demand
H0: no correlation between daily maximum temperature and EMS demand (r = 0)
H1: there is a correlation

In [6]:
r, p_r = stats.pearsonr(summer["tmax"], summer["ems_index"])
rho, p_rho = stats.spearmanr(summer["tmax"], summer["ems_index"])

print(f"Pearson r = {r:.2f}, p = {p_r:.2e}")
print(f"Spearman rho = {rho:.2f}, p = {p_rho:.2e}")

Pearson r = 0.51, p = 1.52e-56
Spearman rho = 0.49, p = 2.11e-51


### Test 4 – Differences across temperature bands
H0: mean EMS demand is the same in all temperature bands
H1: at least one band differs

In [8]:
band_groups = [group["ems_index"] for band, group in summer.groupby("temp_band", observed=True)]

f_stat, p_anova = stats.f_oneway(*band_groups)
h_stat, p_kruskal = stats.kruskal(*band_groups)

print(f"ANOVA: F = {f_stat:.1f}, p = {p_anova:.2e}")
print(f"Kruskal-Wallis: H = {h_stat:.1f}, p = {p_kruskal:.2e}")

ANOVA: F = 33.9, p = 2.67e-46
Kruskal-Wallis: H = 198.7, p = 1.20e-38


### Test 5 – Hot days and weekdays
Sundays are about 4% quieter (EDA). If hot days fell mostly on Sundays or mostly on weekdays, Test 1 would be biased.
H0: hot days are independent of the weekday
H1: hot days are more common on some weekdays

In [10]:
weekday_table = pd.crosstab(summer["weekday"], summer["hot_day"])
print(weekday_table)

chi2, p_chi, dof, expected = stats.chi2_contingency(weekday_table)
print(f"\nChi-square = {chi2:.2f}, p = {p_chi:.2f}, Cramér's V = {association(weekday_table, method='cramer'):.2f}")

hot_day      0   1
weekday           
Friday      94  23
Monday     100  17
Saturday    94  25
Sunday      94  23
Thursday    92  24
Tuesday    100  18
Wednesday   93  25

Chi-square = 3.56, p = 0.74, Cramér's V = 0.07


### Test 6 – Hot days without a heat warning vs. normal days
H0: mean demand on hot days without a warning ≤ normal days without a warning
H1: hot days without a warning are busier

In [11]:
hot_no_warning = summer[(summer["hot_day"] == 1) & (summer["heat_warning"] == 0)]["ems_index"]
normal_no_warning = summer[(summer["hot_day"] == 0) & (summer["heat_warning"] == 0)]["ems_index"]

compare_groups(hot_no_warning, normal_no_warning, "Hot, no warning", "Normal days")

Hot, no warning: n = 72, mean = 103.8
Normal days: n = 660, mean = 98.7
Difference: 5.1 index points, 95% CI [3.7, 6.6]
t = 6.97, p-value = 2.66e-10
Decision: reject H0


### Test 7 – At similar temperatures, are warned days busier?
Compare only days between 30 and 34°C, so temperature is roughly equal.
H0: no difference between warned and unwarned days
H1: there is a difference (two-sided)

In [12]:
band_30_34 = summer[(summer["tmax"] >= 30) & (summer["tmax"] < 34)]

print(band_30_34.groupby("heat_warning").agg(
    days=("tmax", "size"),
    avg_tmax=("tmax", "mean"),
    avg_index=("ems_index", "mean")
).round(1))

compare_groups(band_30_34[band_30_34["heat_warning"] == 1]["ems_index"],
               band_30_34[band_30_34["heat_warning"] == 0]["ems_index"],
               "Warning", "No warning", alternative="two-sided")

              days  avg_tmax  avg_index
heat_warning                           
0               70      31.1      103.8
1               49      32.2      106.3
No warning: n = 70, mean = 103.8
Difference: 2.5 index points, 95% CI [0.4, 4.7]
t = 2.28, p-value = 2.49e-02
Decision: reject H0


### Test 8 – The day after a hot day
Only days that are not hot themselves, so heatwave streaks are not counted twice.
H0: demand on the first non-hot day after a hot day ≤ other normal days
H1: it is higher

In [13]:
after_hot = summer[(summer["hot_day"] == 0) & (summer["prev_day_hot"] == 1)]["ems_index"]
other_normal = summer[(summer["hot_day"] == 0) & (summer["prev_day_hot"] == 0)]["ems_index"]

compare_groups(after_hot, other_normal, "Day after a hot day", "Other normal days")

Day after a hot day: n = 75, mean = 101.1
Other normal days: n = 592, mean = 98.4
Difference: 2.6 index points, 95% CI [0.9, 4.4]
t = 2.90, p-value = 2.35e-03
Decision: reject H0


In [14]:
compare_groups(summer[summer["hot_day"] == 1]["cpr_index"],
               summer[summer["hot_day"] == 0]["cpr_index"],
               "CPR alerts, hot days", "CPR alerts, normal days")

CPR alerts, hot days: n = 155, mean = 105.2
CPR alerts, normal days: n = 667, mean = 98.8
Difference: 6.5 index points, 95% CI [1.5, 11.4]
t = 2.55, p-value = 5.73e-03
Decision: reject H0


## Summary of hypothesis tests

| # | Question | Test | Result | Decision |
|---|---|---|---|---|
| 1 | Hot days busier? | Welch t-test (one-sided) | +6.7 pts, 95% CI [5.7, 7.8], d = 1.09 | Reject H0 |
| 2 | Consistent across years? | Sign test | 9 of 9 summers, p = 0.002 | Reject H0 |
| 3 | Demand rises with temperature? | Pearson / Spearman | r = 0.51 / 0.49 | Reject H0 |
| 4 | Differences across temperature bands? | ANOVA / Kruskal-Wallis | p < 0.001 | Reject H0 |
| 5 | Hot days evenly spread across weekdays? | Chi-square | p = 0.74 | Cannot reject H0 (good: no bias) |
| 6 | Hot days without warning busier? | Welch t-test | +5.1 pts, 95% CI [3.7, 6.6] | Reject H0 |
| 7 | Warned vs. unwarned at 30–34°C | Welch t-test (two-sided) | +2.5 pts, 95% CI [0.4, 4.7] | Reject H0 (marginal) |
| 8 | Day after a hot day | Welch t-test | +2.6 pts, 95% CI [0.9, 4.4] | Reject H0 |
| 9 | CPR alerts on hot days | Welch t-test | +6.5 pts, 95% CI [1.5, 11.4] | Reject H0 (uncertain) |

**Caveats**
- Consecutive days are not independent (heatwaves), so t-test p-values are optimistic. Test 2 (one observation per summer) is the most robust evidence.
- Many tests increase the chance of a false positive. Tests 1–4 and 6 are far below any threshold; Tests 7–9 are closer and should be read as indicative.
- All results are associations, not proof of causation.

## Main findings

**Hot days put measurable extra pressure on Berlin's emergency medical services.**

1. **Hot days are busier.** On days of 30°C or more, emergency medical demand is about **7% higher** than on other summer days (95% CI 5.7–7.8 points; large effect, Cohen's d = 1.09).
2. **It happens every year.** Hot days were busier in **all 9 summers from 2018 to 2026** (p = 0.002), so the effect isn't driven by one exceptional year.
3. **Demand rises steadily with temperature.** There's no sudden jump at 30°C: demand climbs gradually, from about 5% below average on days under 20°C to about 8% above on days over 34°C (r = 0.51).
4. **Heat warnings miss many demanding days.** **72 of 155 hot days had no official DWD heat warning**, yet demand on those days was still about **5% higher** than on normal days (95% CI 3.7–6.6). Warned days are busier still, partly because they're hotter and partly because the warnings capture more than daytime heat (warm nights, consecutive hot days).
5. **The effect lingers.** On the first cooler day after a hot day, demand is still about **2–3% higher** than normal.
6. **Severe cases may rise too.** Suspected cardiac arrests (CPR alerts) are about 6% higher on hot days, but with high uncertainty because daily numbers are small.

**What this means:** for emergency-service planning, temperature itself, not only an official heat warning, is a useful early signal for higher demand. This matters more as climate change brings more hot days to Berlin.

**Limitations:** these are associations, not proof of causation. One weather station represents the whole city, consecutive hot days are not independent (so the yearly consistency check is the most robust evidence), and 2026 data is preliminary and has missing days.